In [13]:
import os
import glob
import RNA
import statistics
import csv 

rna_fa = glob.glob("/scr/velara/bioinf2/TBI.Project.AG/results/results_bedtools/folding_rna_*.fa")

for fa_file in rna_fa:
    output_tsv = f"{os.path.splitext(fa_file)[0]}_mfe.tsv"

    md = RNA.md()
    mfe_values = []
    sequences = []

    with open(fa_file, "r") as f:
        content = f.read().strip().split(">")
        for entry in content:
            if not entry:
                continue
            lines = entry.split("\n")
            seq = "".join(lines[1:]).strip()
            if seq:
                sequences.append(seq)

    for i, seq in enumerate(rna_fa, 1):
        fc = RNA.fold_compound(seq, md)
        structure, mfe = fc.mfe()
        mfe_values.append(mfe)

    with open(output_tsv, "w", newline='') as tsvfile:
        writer = csv.writer(tsvfile, delimiter = '\t')
        writer.writerow(["ID", "Sequence", "mfe", "structure"])
        for i, (seq, mfe, structure) in enumerate(zip(sequences, mfe_values, structure), 1):
            writer.writerow([f"sequence_{i}", seq, mfe, structure])


In [10]:
input_fasta = "/scr/velara/bioinf2/TBI.Project.AG/results/results_bedtools/folding_rna_z1.fa"
output_tsv = input_fasta.replace(".fa", "_mfe.tsv")

sequences = []
results = []

with open(input_fasta, "r") as f:
    for line in f:
        line = line.strip()
        if line.startswith(">"):
            seq_id = line[1:]
        elif line:
            sequences.append((seq_id, line))

for seq_id, seq in sequences:
    fc = RNA.fold_compound(seq)
    structure, mfe = fc.mfe()
    results.append((seq_id, mfe, structure, seq))

with open(output_tsv, "w", newline="") as tsvfile:
    writer = csv.writer(tsvfile, delimiter="\t")
    writer.writerow(["id", "mfe", "structure", "seq"])
    writer.writerows(results)
        

In [11]:
input_fasta = "/scr/velara/bioinf2/TBI.Project.AG/results/results_bedtools/folding_rna_z3.fa"
output_tsv = input_fasta.replace(".fa", "_mfe.tsv")

sequences = []
results = []

with open(input_fasta, "r") as f:
    for line in f:
        line = line.strip()
        if line.startswith(">"):
            seq_id = line[1:]
        elif line:
            sequences.append((seq_id, line))

for seq_id, seq in sequences:
    fc = RNA.fold_compound(seq)
    structure, mfe = fc.mfe()
    results.append((seq_id, mfe, structure, seq))

with open(output_tsv, "w", newline="") as tsvfile:
    writer = csv.writer(tsvfile, delimiter="\t")
    writer.writerow(["id", "mfe", "structure", "seq"])
    writer.writerows(results)

In [12]:
input_fasta = "/scr/velara/bioinf2/TBI.Project.AG/results/results_bedtools/folding_rna_z5.fa"
output_tsv = input_fasta.replace(".fa", "_mfe.tsv")

sequences = []
results = []

with open(input_fasta, "r") as f:
    for line in f:
        line = line.strip()
        if line.startswith(">"):
            seq_id = line[1:]
        elif line:
            sequences.append((seq_id, line))

for seq_id, seq in sequences:
    fc = RNA.fold_compound(seq)
    structure, mfe = fc.mfe()
    results.append((seq_id, mfe, structure, seq))

with open(output_tsv, "w", newline="") as tsvfile:
    writer = csv.writer(tsvfile, delimiter="\t")
    writer.writerow(["id", "mfe", "structure", "seq"])
    writer.writerows(results)

In [19]:
import random
import numpy as numpy
import pandas as pd 

input_fasta = "/scr/velara/bioinf2/TBI.Project.AG/results/results_bedtools/folding_rna_z1.fa"
output_dir = "/scr/velara/bioinf2/TBI.Project.AG/results/results_bedtools"
n_shuf = 1000
top_n = 10

sequences = []


with open(input_fasta, 'r') as f:
    header = None
    for line in f:
        line = line.strip()
        if line.startswith('>::'):
            header = line[3:]
        elif line:
            sequences.append((header, line))
            if len(sequences) >= top_n:
                break

for i, (seq_id, seq) in enumerate(sequences, 1):
    out_path = os.path.join(output_dir, f"{seq_id}.fa")
    with open(out_path, 'w') as out:
        out.write(f">{seq_id}_original\n{seq}\n")
        for j in range(n_shuf):
            shuffled = list(seq)
            random.shuffle(shuffled)
            shuffled_seq = ''.join(shuffled)
            out.write(f">{seq_id}_shuffle_{j+1}\n{shuffled_seq}\n")



In [36]:
input_folder = '/scr/velara/bioinf2/TBI.Project.AG/results/results_bedtools/results_ribo_structure_z1'
output_folder = '/scr/velara/bioinf2/TBI.Project.AG/results/results_bedtools/results_ribo_structure_z1'

fasta_files = [f for f in os.listdir(input_folder) if f.endswith('.fa')]

for file in fasta_files:
    fasta_path = os.path.join(input_folder, file)
    output_tsv = os.path.join(output_folder, file.replace('.fa', '.tsv'))

    sequences = []
    with open(fasta_path, 'r') as f:
        header = None
        for line in f:
            line = line.strip()
            if line.startswith('>'):
                header = line[1:]
            elif line:
                sequences.append((header, line))
    if not sequences:
        continue

    mfes = []
    for seq_id, seq in sequences:
        fc = RNA.fold_compound(seq)
        structure, mfe_value = fc.mfe()
        mfes.append(mfe_value)

    original_mfe = mfes[0]
    shuffled_mfes = mfes[1:]
    mean_shuf = numpy.mean(shuffled_mfes)
    std_shuf = numpy.std(shuffled_mfes)
    z_score = (original_mfe - mean_shuf) / std_shuf if std_shuf != 0 else 0

    with open(output_tsv, 'w', newline ='') as tsvfile:
        writer = csv.writer(tsvfile, delimiter = '\t')
        writer.writerow(["seq", 'mfe', 'z_score'])
        writer.writerow([sequences[0][1], original_mfe, z_score])
        for (seq_id, seq), mfe in zip(sequences[1:], mfes[1:]):
            writer.writerow([seq, mfe, ''])